# Web Scraping
---
In today's lab, we are going to download data from the internet using an API. API stands for **a**pplication **p**rogramming **i**nterface. Companies often create APIs as a way to allow users to more directly interact with their servers to retrieve data. Today, we are going to be using Twitter's API to download tweets to get some experience with large data.

In [4]:
# Run this cell to set up your notebook
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zipfile
import warnings
import twitter_utils as tu

# Ensure that Pandas shows at least 280 characters in columns, so we can see full tweets
pd.set_option('max_colwidth', 280)

%matplotlib inline
import seaborn as sns
sns.set()
sns.set_context("talk")
import re
import json

## Setup
---
For this lab, we will be importing utility functions to interact with Twitter's API. Underneath the hood, these utility functions use the `tweepy` package, which is the how you can interface with Twitter's API using `python`. First, we need to install `tweepy` so that our utility functions will be able to use use it. If you are interested in seeing the code for these functions, you can look in the `twitter_utils.py` file in this folder.

In [5]:
!pip install tweepy

Twitter requires you to have authentication keys to access their API.  To get your keys, you'll have to sign up for a Twitter developer account. Note that **anyone who has your authentication keys can post as you**. In order to protect your keys, you will be storing them in a separate file, which we have called `keys.json`, and reading them into this notebook from that file. Also note that **Twitter limits developers to a certain rate of data requests**. This means that if you make too many API calls in a short period of time, Twitter may block you from retrieving data for a certain period of time. Avoid rerunning cells that retrieve new tweets.

Follow the instructions below to get your Twitter API keys.  **Read the instructions completely before starting.**

1. [Create a Twitter account](https://twitter.com).  You can use an existing account if you have one; if you prefer to not do this assignment under your regular account, feel free to create a throw-away account.
2. Under account settings, add your phone number to the account.
3. [Create a Twitter developer account](https://dev.twitter.com/resources/signup) by clicking the 'Apply' button on the top right of the page. Attach it to your Twitter account. You'll have to fill out a form describing what you want to do with the developer account. Explain that you are doing this for a class at UC Berkeley and that you don't know exactly what you're building yet and just need the account to get started. These applications are approved by some sort of AI system, so it doesn't matter exactly what you write.
4. Once you're logged into your developer account, [create an application for this assignment](https://apps.twitter.com/app/new).  You can call it whatever you want, and you can write any URL when it asks for a web site.  You don't need to provide a callback URL.
5. On the page for that application, find your Consumer Key and Consumer Secret.
6. On the same page, create an Access Token.  Record the resulting Access Token and Access Token Secret.
7. Edit the file `keys.json` in the same folder as this file and replace the placeholders with your keys.

Now you should be all ready to go! Let's test that you have correctly set up your developer account and the `keys.json` folder. The following cell loads your keys into this notebook, then validates them with the Twitter API. It should display your Twitter username without any warnings.

In [6]:
import json
key_file = "monicas_keys.json" #Change to "keys.json" in final version.

# Loading your keys from keys.json
with open(key_file) as f:
    keys = json.load(f)
# if you print or view the contents of keys be sure to delete the cell!

# Validate keys
tu.validate_authentication(keys)

JSONDecodeError: Expecting value: line 3 column 23 (char 73)

If you are getting any errors in this cell, ask a TA for help. If you do not have valid keys, you will not be able to use the API.

## Download Tweets
---
Now we should be ready to download some tweets! In the following cell, we use one of the utility functions to download recent tweets with the hashtag "#data".

In [49]:
data = tu.download_recent_tweets_by_hashtag(hashtag = "snow",
                                           keys = keys)

NameError: name 'TweepError' is not defined

We now have `data` assigned to a list with each element corresponding to a tweet. Let's examine one of these elements to get a better understanding of our data.

In [15]:
try:
    auth = tweepy.OAuthHandler(keys["consumer_key"], keys["consumer_secret"])
    auth.set_access_token(keys["access_token"], keys["access_token_secret"])
    api = tweepy.API(auth)
    tweets = [t._json for t in tweepy.Cursor(api.search,geocode = "37.8716,122.2727,1km",
                       lang="en").items()]
except TweepError as e:
    logging.warning("There was a Tweepy error. Double check your API keys and try again.")
    logging.warning(e)


NameError: name 'TweepError' is not defined

In [18]:
tweets

NameError: name 'tweets' is not defined

In [19]:
data[0]

{'created_at': 'Mon Apr 08 07:28:32 +0000 2019',
 'id': 1115154444126228481,
 'id_str': '1115154444126228481',
 'text': '#Server #Colocation. Clients typically choose to have their own servers hosted in our #data #center.\nFor More Info:… https://t.co/m8fZRST2uH',
 'truncated': True,
 'entities': {'hashtags': [{'text': 'Server', 'indices': [0, 7]},
   {'text': 'Colocation', 'indices': [8, 19]},
   {'text': 'data', 'indices': [86, 91]},
   {'text': 'center', 'indices': [92, 99]}],
  'symbols': [],
  'user_mentions': [],
  'urls': [{'url': 'https://t.co/m8fZRST2uH',
    'expanded_url': 'https://twitter.com/i/web/status/1115154444126228481',
    'display_url': 'twitter.com/i/web/status/1…',
    'indices': [117, 140]}]},
 'metadata': {'iso_language_code': 'en', 'result_type': 'recent'},
 'source': '<a href="https://www.hootsuite.com" rel="nofollow">Hootsuite Inc.</a>',
 'in_reply_to_status_id': None,
 'in_reply_to_status_id_str': None,
 'in_reply_to_user_id': None,
 'in_reply_to_user_id_st

This is an example of another `python` data structure called a *dictionary*. Dictionaries store values by associating them with a *key* rather than by an integer index. You can index into a dictionary using bracket notation just like a list. For example

In [24]:
d = {'a': 1,
    'b': 2,
    'c': 3}
d['a']

1

The dictionary we were looking at above is a little bit hard to interpret because there dictionaries nested inside of some our keys. We can look only at the first level of keys in our dictionary by using the `.keys()` method.

In [36]:
data[0].keys()

dict_keys(['created_at', 'id', 'id_str', 'text', 'truncated', 'entities', 'metadata', 'source', 'in_reply_to_status_id', 'in_reply_to_status_id_str', 'in_reply_to_user_id', 'in_reply_to_user_id_str', 'in_reply_to_screen_name', 'user', 'geo', 'coordinates', 'place', 'contributors', 'retweeted_status', 'is_quote_status', 'retweet_count', 'favorite_count', 'favorited', 'retweeted', 'possibly_sensitive', 'lang'])

The most relevant keys for what we will be doing today are `'created at'`, `'text'`, `''`

In [35]:
data[25]['retweeted_status']

{'contributors': None,
 'coordinates': None,
 'created_at': 'Fri Mar 29 18:35:12 +0000 2019',
 'entities': {'hashtags': [{'indices': [0, 3], 'text': 'AI'},
   {'indices': [51, 56], 'text': 'data'},
   {'indices': [83, 92], 'text': 'security'}],
  'symbols': [],
  'urls': [{'display_url': 'twitter.com/i/web/status/1…',
    'expanded_url': 'https://twitter.com/i/web/status/1111698338116710400',
    'indices': [106, 129],
    'url': 'https://t.co/RKt41pEdxd'}],
  'user_mentions': []},
 'favorite_count': 2,
 'favorited': False,
 'geo': None,
 'id': 1111698338116710400,
 'id_str': '1111698338116710400',
 'in_reply_to_screen_name': None,
 'in_reply_to_status_id': None,
 'in_reply_to_status_id_str': None,
 'in_reply_to_user_id': None,
 'in_reply_to_user_id_str': None,
 'is_quote_status': False,
 'lang': 'en',
 'metadata': {'iso_language_code': 'en', 'result_type': 'recent'},
 'place': None,
 'possibly_sensitive': False,
 'retweet_count': 7,
 'retweeted': False,
 'source': '<a href="https://ww

In [28]:
retweeted = [];
for i in data:
    if i['retweet_count'] > 0:
        retweeted.append(i)

In [29]:
len(data) - len(retweeted)

84

Knowing that we have a list doesn't actually tell us very much about the rest of our data. Let's examine one of the elements of our 

In [11]:
df = pd.DataFrame(data)
df['date'] = df['created_at'].apply(lambda d: pd.datetime.strptime(str(d),"%a %b %d %X %z %Y"))
df['month'] = df['date'].apply(lambda d: d.strftime('%Y-%m-%d %X'))

In [12]:
list(df)

['contributors',
 'coordinates',
 'created_at',
 'entities',
 'extended_entities',
 'favorite_count',
 'favorited',
 'geo',
 'id',
 'id_str',
 'in_reply_to_screen_name',
 'in_reply_to_status_id',
 'in_reply_to_status_id_str',
 'in_reply_to_user_id',
 'in_reply_to_user_id_str',
 'is_quote_status',
 'lang',
 'metadata',
 'place',
 'possibly_sensitive',
 'quoted_status',
 'quoted_status_id',
 'quoted_status_id_str',
 'retweet_count',
 'retweeted',
 'retweeted_status',
 'source',
 'text',
 'truncated',
 'user',
 'date',
 'month']

In [13]:
hashtag_df = df[['id', 'retweet_count', 'source', 'ful\ 'date']]
hashtag_df.columns = ['id', 'retweet_count', 'source', 'text', 'date'] #can also rename columns
hashtag_df.head() #take a peep at the first 5 rows

SyntaxError: invalid syntax (<ipython-input-13-1c19d2b5b45e>, line 1)

In [41]:
tweets_and_locations = list()
for tweet in data:
    new_entry = {}
    new_entry['tweet'] = tweet['text']
    new_entry['location'] = tweet['user']['location']
    tweets_and_locations.append(new_entry)
    

In [43]:
aaa = pd.DataFrame(tweets_and_locations)
aaa

,location,tweet
0,Nigeria,RT @powwerme: The Story. \nDrops 4/9.\n#socialimpact #entrepreneurs #founder #strategy #vision #businessdevelopment #oakland #bayarea #berkel…
1,"Cardiff, Wales","RT @wconcerthall: Today, @BBCNOW plays a #Puw and a #Boden world premieres #Hoddinott and #Berkeley in #Cardiff https://t.co/Et09TR5j0K #w…"
2,"Amarillo, TX",RT @SquirrelFabric: 😍😍😍 Fresh @thefarmersdaughterfibers Squish Sock and Foxy Lady are on the shelf! Come squish it today till 5!\n#sockyarn…
3,"Barcelona, Catalonia","Today, @BBCNOW plays a #Puw and a #Boden world premieres #Hoddinott and #Berkeley in #Cardiff https://t.co/Et09TR5j0K #wch"
4,https://www.facebook.com/viari,Viari’s https://t.co/ZTftGZAJbY can help you makeover a new wardrobe with just a new bag instead of many dresses. C… https://t.co/Z1Mxq4Tp0o
5,,RT @powwerme: The Story. \nDrops 4/9.\n#socialimpact #entrepreneurs #founder #strategy #vision #businessdevelopment #oakland #bayarea #berkel…
6,,RT @powwerme: The Story. \nDrops 4/9.\n#socialimpact #entrepreneurs #founder #strategy #vision #businessdevelopment #oakland #bayarea #berkel…
7,"San Francisco, CA","RT @WHELANSSMOSHOP: 🚨 4/20 GIVEAWAY‼️\nEvery $20 gets you a raffle ticket to the Live Raffle Drawing Saturday 6pm \n\n#Berkeley, Ca \n1st @his…"
8,"Berkeley, CA",In every walk with nature one receives far more than he seeks. ~ John Muir #cherryblossoms #berkeley #california https://t.co/uFmmCs2lL1
9,From Coast to Coast,"Throwback to The Story So Far's Holiday Show\n12/22/2018\nUC Theatre\nBerkeley, CA\n#TheStorySoFar #TSSF… https://t.co/H7PQbUhEF3"


Source: http://www.mikaelbrunila.fi/2017/03/27/scraping-extracting-mapping-geodata-twitter/

In [20]:
#Create dictionary to later be stored as JSON. All data will be included
# in the list 'data'
users_with_geodata = {
    "data": []
}
all_users = []
total_tweets = 0
geo_tweets  = 0
for tweet in data:
    if tweet['user']['id']:
        total_tweets += 1 
        user_id = tweet['user']['id']
        if user_id not in all_users:
            all_users.append(user_id)

            #Give users some data to find them by. User_id listed separately 
            # to make iterating this data later easier
            user_data = {
                "user_id" : tweet['user']['id'],
                "features" : {
                    "name" : tweet['user']['name'],
                    "id": tweet['user']['id'],
                    "screen_name": tweet['user']['screen_name'],
                    "tweets" : 1,
                    "location": tweet['user']['location'],
                }
            }
            #Iterate through different types of geodata to get the variable primary_geo
            if tweet['coordinates']:
                user_data["features"]["primary_geo"] = str(tweet['coordinates'][tweet['coordinates'].keys()[1]][1]) + ", " + str(tweet['coordinates'][tweet['coordinates'].keys()[1]][0])
                user_data["features"]["geo_type"] = "Tweet coordinates"
            elif tweet['place']:
                user_data["features"]["primary_geo"] = tweet['place']['full_name'] + ", " + tweet['place']['country']
                user_data["features"]["geo_type"] = "Tweet place"
            else:
                user_data["features"]["primary_geo"] = tweet['user']['location']
                user_data["features"]["geo_type"] = "User location"
            #Add only tweets with some geo data to .json. Comment this if you want to include all tweets.
            if user_data["features"]["primary_geo"]:
                users_with_geodata['data'].append(user_data)
                geo_tweets += 1

        #If user already listed, increase their tweet count
        elif user_id in all_users:
            for user in users_with_geodata["data"]:
                if user_id == user["user_id"]:
                    user["features"]["tweets"] += 1

#Count the total amount of tweets for those users that had geodata            
for user in users_with_geodata["data"]:
    geo_tweets = geo_tweets + user["features"]["tweets"]

# Save data to JSON file
with open('geolocation_tweets.json', 'w') as fout:
    fout.write(json.dumps(users_with_geodata, indent=4))

The file included 425 unique users who tweeted with or without geo data
The file included 325 unique users who tweeted with geo data, including 'location'
The users with geo data tweeted 816 out of the total 617 of tweets.


NameError: name 'data' is not defined